In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
print(sys.executable)

/usr/bin/python3


In [3]:
import os
os.getcwd()

'/content'

In [4]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Number of GPUs:", torch.cuda.device_count())

CUDA available: True
GPU name: Tesla T4
Number of GPUs: 1


Installations

In [19]:
!pip install requests tqdm

cell 2 - XenoCanto API 

In [5]:
import os

# Xeno-canto API key (keep this private)
os.environ["XENOCANTO_API_KEY"] = "66a66df58dca4ebd5610998429c48ad70d89ee5b"


Selection of species

In [6]:
import os

# Base project directory
BASE_DIR = "/content/drive/MyDrive/bird_audio_detector"

# All downloaded audio will go inside this folder
DATASET_DIR = os.path.join(BASE_DIR, "input")

# Create input folder if it doesn't exist
os.makedirs(DATASET_DIR, exist_ok=True)

# Updated species list (5 old + 10 new)
TARGET_SPECIES = [
    # Existing species
    "Corvus splendens",
    "Acridotheres tristis",
    "Passer domesticus",
    "Columba livia",
    "Psittacula krameri",

    # New species
    "Pycnonotus cafer",
    "Eudynamys scolopaceus",
    "Copsychus saularis",
    "Turdoides striata",
    "Halcyon smyrnensis",
    "Alcedo atthis",
    "Pavo cristatus",
    "Gracula religiosa",
    "Dicrurus macrocercus",
    "Streptopelia decaocto"
]

MIN_DURATION = 2
MAX_DURATION = 10
ALLOWED_QUALITY = ["A", "B"]
MAX_FILES_PER_SPECIES = 50

Fetching of audio recodings from xenocanto

In [ ]:
import requests
import os

API_KEY = os.environ.get("XENOCANTO_API_KEY")

def fetch_xenocanto_page(species_name, page):
    genus, species = species_name.split()
    query = f"gen:{genus} sp:{species}"

    url = "https://xeno-canto.org/api/3/recordings"
    params = {
        "query": query,
        "key": API_KEY,
        "per_page": 100,
        "page": page
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()


Testing the successful retrival of audio recordings

In [8]:
test = fetch_xenocanto_page("Corvus splendens", 1)

print("Recordings:", test["numRecordings"])
print("Pages:", test["numPages"])
print("First ID:", test["recordings"][0]["id"])
print("File URL:", test["recordings"][0]["file"])

Recordings: 219
Pages: 3
First ID: 1046832
File URL: https://xeno-canto.org/1046832/download


checks the recording is valid or not

In [9]:
def is_valid_recording(record):
    try:
        mins, secs = record["length"].split(":")
        duration = int(mins) * 60 + int(secs)

        return (
            MIN_DURATION <= duration <= MAX_DURATION
            and record["q"] in ALLOWED_QUALITY
        )
    except:
        return False


downloading function

In [10]:
import os
from tqdm import tqdm

def download_species_audio(species_name):
    species_folder = species_name.replace(" ", "_")
    save_path = os.path.join(DATASET_DIR, species_folder)
    os.makedirs(save_path, exist_ok=True)

    downloaded = 0
    page = 1

    while downloaded < MAX_FILES_PER_SPECIES:
        data = fetch_xenocanto_page(species_name, page)

        for record in data["recordings"]:
            if downloaded >= MAX_FILES_PER_SPECIES:
                break

            if not is_valid_recording(record):
                continue

            # ✅ CORRECT URL HANDLING
            file_url = record["file"]
            if file_url.startswith("//"):
                audio_url = "https:" + file_url
            elif file_url.startswith("http"):
                audio_url = file_url
            else:
                continue

            file_name = f"XC{record['id']}.mp3"
            file_path = os.path.join(save_path, file_name)

            if os.path.exists(file_path):
                continue

            r = requests.get(audio_url, stream=True)
            with open(file_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024):
                    if chunk:
                        f.write(chunk)

            downloaded += 1

        if page >= int(data["numPages"]):
            break
        page += 1

    print(f"✅ {species_name}: {downloaded} files downloaded")


In [11]:
for species in TARGET_SPECIES:
    print(f"\nDownloading trial data for: {species}")
    download_species_audio(species)



✅ Corvus splendens: 0 files downloaded

✅ Acridotheres tristis: 0 files downloaded

✅ Passer domesticus: 50 files downloaded

✅ Columba livia: 0 files downloaded

✅ Psittacula krameri: 24 files downloaded

✅ Pycnonotus cafer: 0 files downloaded

✅ Eudynamys scolopaceus: 34 files downloaded

✅ Copsychus saularis: 0 files downloaded

✅ Turdoides striata: 0 files downloaded

✅ Halcyon smyrnensis: 3 files downloaded

✅ Alcedo atthis: 50 files downloaded

✅ Pavo cristatus: 50 files downloaded

✅ Gracula religiosa: 0 files downloaded

✅ Dicrurus macrocercus: 10 files downloaded

✅ Streptopelia decaocto: 50 files downloaded


cell 0


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


New structure

cell 1

In [25]:
!pip install requests tqdm

cell 2

In [26]:
import os

# Xeno-canto API key (keep this private)
os.environ["XENOCANTO_API_KEY"] = "66a66df58dca4ebd5610998429c48ad70d89ee5b"


cell 3

In [27]:
import os

# Base project directory
BASE_DIR = "/content/drive/MyDrive/bird_audio_detector"

# All downloaded audio will go inside this folder
DATASET_DIR = os.path.join(BASE_DIR, "input")

# Create input folder if it doesn't exist
os.makedirs(DATASET_DIR, exist_ok=True)

# Species with large datasets on Xeno-Canto
TARGET_SPECIES = [
    "Passer domesticus",      # House Sparrow
    "Eudynamys scolopaceus",  # Asian Koel
    "Alcedo atthis",          # Common Kingfisher
    "Pavo cristatus",         # Indian Peafowl
    "Streptopelia decaocto",  # Eurasian Collared Dove
    "Turdus merula",          # Common Blackbird
    "Parus major",            # Great Tit
    "Erithacus rubecula",     # European Robin
    "Luscinia megarhynchos",  # Nightingale
    "Cyanistes caeruleus",    # Blue Tit
    "Sturnus vulgaris",       # Common Starling
    "Phylloscopus collybita", # Chiffchaff
    "Troglodytes troglodytes",# Eurasian Wren
    "Fringilla coelebs",      # Chaffinch
    "Sylvia atricapilla"      # Blackcap
]

MIN_DURATION = 2
MAX_DURATION = 10
ALLOWED_QUALITY = ["A", "B"]
MAX_FILES_PER_SPECIES = 50

cell 4

In [28]:
import requests

API_KEY = os.environ.get("XENOCANTO_API_KEY")

def fetch_xenocanto_page(species_name, page):
    genus, species = species_name.split()
    query = f"gen:{genus} sp:{species}"

    url = "https://xeno-canto.org/api/3/recordings"
    params = {
        "query": query,
        "key": API_KEY,
        "per_page": 100,
        "page": page
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

cell 5

In [29]:
test = fetch_xenocanto_page("Corvus splendens", 1)

print("Recordings:", test["numRecordings"])
print("Pages:", test["numPages"])
print("First ID:", test["recordings"][0]["id"])
print("File URL:", test["recordings"][0]["file"])

Recordings: 219
Pages: 3
First ID: 1046832
File URL: https://xeno-canto.org/1046832/download


cell 6

In [30]:
def is_valid_recording(record):
    try:
        mins, secs = record["length"].split(":")
        duration = int(mins) * 60 + int(secs)

        return (
            MIN_DURATION <= duration <= MAX_DURATION
            and record["q"] in ALLOWED_QUALITY
        )
    except:
        return False

cell 7

In [31]:
from tqdm import tqdm

def download_species_audio(species_name):
    species_folder = species_name.replace(" ", "_")
    save_path = os.path.join(DATASET_DIR, species_folder)
    os.makedirs(save_path, exist_ok=True)

    downloaded = 0
    page = 1

    while downloaded < MAX_FILES_PER_SPECIES:
        data = fetch_xenocanto_page(species_name, page)

        for record in data["recordings"]:
            if downloaded >= MAX_FILES_PER_SPECIES:
                break

            if not is_valid_recording(record):
                continue

            file_url = record["file"]

            if file_url.startswith("//"):
                audio_url = "https:" + file_url
            elif file_url.startswith("http"):
                audio_url = file_url
            else:
                continue

            file_name = f"XC{record['id']}.mp3"
            file_path = os.path.join(save_path, file_name)

            if os.path.exists(file_path):
                continue

            try:
                r = requests.get(audio_url, stream=True, timeout=30)
                r.raise_for_status()

                with open(file_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)

                downloaded += 1

            except Exception as e:
                print(f"Failed to download {audio_url}: {e}")
                continue

        if page >= int(data["numPages"]):
            break

        page += 1

    print(f"✅ {species_name}: {downloaded} files downloaded")

cell 8

In [32]:
for species in TARGET_SPECIES:
    print(f"\nDownloading trial data for: {species}")
    download_species_audio(species)


✅ Passer domesticus: 50 files downloaded

✅ Eudynamys scolopaceus: 50 files downloaded

✅ Alcedo atthis: 50 files downloaded

✅ Pavo cristatus: 50 files downloaded

✅ Streptopelia decaocto: 50 files downloaded

✅ Turdus merula: 50 files downloaded

✅ Parus major: 50 files downloaded

✅ Erithacus rubecula: 50 files downloaded

✅ Luscinia megarhynchos: 50 files downloaded

✅ Cyanistes caeruleus: 50 files downloaded

✅ Sturnus vulgaris: 50 files downloaded

Failed to download https://xeno-canto.org/507301/download: 500 Server Error: Internal Server Error for url: https://xeno-canto.org/507301/download
✅ Phylloscopus collybita: 50 files downloaded

✅ Troglodytes troglodytes: 50 files downloaded

✅ Fringilla coelebs: 50 files downloaded

✅ Sylvia atricapilla: 50 files downloaded


In [5]:
%cd /content/drive/MyDrive/bird_audio_detector

/content/drive/MyDrive/bird_audio_detector


In [6]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)

In [6]:
!pip install librosa numpy soundfile tqdm scikit-learn


In [5]:
!pip install -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 MB 14.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 134.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 127.2 MB/s eta 0:00:0000:01
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=0ec7ff8011536ff24c2efc8e9ffb6a89964147dfe7321567e48dac49b216d9d3
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
  Attempting uninstall: watchdog
    Found existing installation: watchdog 6.0.0
    Uninstalling watchdog-6.0.0:
      Successfully uninstalled watchdog-6.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages tha

In [11]:
!python main.py

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
E0000 00:00:1774459259.618504    2411 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774459259.625135    2411 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registere

In [10]:
!python eval_prebuilt.py


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
E0000 00:00:1774462679.179423    5035 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774462679.190353    5035 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registere

Test codes

In [24]:
rm -rf input

In [ ]:
import numpy as np

data = np.load("XC79536_source_0.npy")
print(data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

# Load the two separated sources of a single example file
file_0 = "processed/train/Corvus_splendens/XC165854_source_0.npy" # change to a real filename you have
file_1 = "processed/train/Corvus_splendens/XC165854_source_1.npy"

mel_0 = np.load(file_0)
mel_1 = np.load(file_1)

print(f"Shape of Source 0: {mel_0.shape}")
print(f"Value Range: Min {mel_0.min():.2f}, Max {mel_0.max():.2f}")

# Visualize them side-by-side
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
librosa.display.specshow(mel_0, x_axis='time', y_axis='mel', sr=22050)
plt.title('Source 0 (e.g., Bird / Noise)')
plt.colorbar(format='%+2.0f dB')

plt.subplot(1, 2, 2)
librosa.display.specshow(mel_1, x_axis='time', y_axis='mel', sr=22050)
plt.title('Source 1 (e.g., Background / Other Bird)')
plt.colorbar(format='%+2.0f dB')
plt.tight_layout()
plt.show()
